# 🧠 OmniAgent Engine — Developer Prototyping & Benchmarking Sandbox

Welcome to the **OmniAgent Developer Playground**. This notebook demonstrates:
1. **Task Splitting & Hybrid Routing**: Classifying user prompts into **Local SLM (On-Device)** vs **Cloud LLM (Offloaded)**.
2. **Agent Chains & Reasoning Steps**: Visualizing step-by-step reasoning (`THINKING` → `ROUTING` → `PLANNING` → `EXECUTING`).
3. **Local Latency Benchmarking**: Measuring on-device inference speed vs cloud offloading.
4. **Custom Tool Registration**: Writing custom tools for local system automation.

In [ ]:
import sys
from pathlib import Path

# Add agent package to system path
repo_root = Path("..").resolve()
sys.path.insert(0, str(repo_root / "agent"))

from omniagent import TaskRouter, TaskPlanner, AgentExecutor, AgentMemory
from omniagent.providers.local import LocalProvider
import time

## 1. Prototype the Hybrid Task Router
The **TaskRouter** uses multi-signal heuristic complexity scoring to decide whether a prompt is handled by a 1B-3B on-device model or offloaded to Cloud.

In [ ]:
router = TaskRouter(complexity_threshold=0.55)

test_prompts = [
    "Summarize my unread notifications from the last hour",
    "Audit this directory for exposed API credentials",
    "Format this CSV document and clean whitespace",
    "Derive the mathematical proof for quantum entanglement entropy and simulate eigenstates",
    "Develop a comprehensive 50-page enterprise financial strategy architecture"
]

print(f"{'Task Prompt':<65} | {'Decision':<10} | {'Score':<6}")
print("-" * 87)
for prompt in test_prompts:
    res = router.route(prompt)
    print(f"{prompt[:62] + '...':<65} | {res.decision.value:<10} | {res.complexity_score:<6.3f}")

## 2. Benchmark Local C++ Engine Latency
Measure on-device execution time through the compiled native core.

In [ ]:
provider = LocalProvider()

async def benchmark_inference(prompt: str, iterations: int = 5):
    latencies = []
    for i in range(iterations):
        start = time.perf_counter()
        out = await provider.generate(prompt)
        latencies.append((time.perf_counter() - start) * 1000)
    
    avg = sum(latencies) / len(latencies)
    print(f"Benchmark prompt: '{prompt}'")
    print(f"Avg Latency: {avg:.2f} ms over {iterations} runs")
    print(f"Output: {out}")

import asyncio
await benchmark_inference("Audit local security policies and secret keys")

## 3. End-to-End Agent Execution
Execute a complete plan-route-execute cycle.

In [ ]:
executor = AgentExecutor()
result = await executor.run("Check system disk and list current files")
print("\n--- Execution Result ---")
print(result)